# Steps 9-10: Pseudo-Labels + Confidence Filtering

**Objectives**:
1. Generate pseudo-labels from fine-tuned model on unlabeled frames
2. Estimate confidence (max softmax probability + augmentation agreement)
3. Analyze pseudo-label quality on validation set (where GT exists)
4. Sweep confidence thresholds
5. Visualize pseudo-labels before/after filtering

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import ACDCSegDataset, get_val_transforms
from src.segmentation_model import SegmentationUNet
from src.pseudo_labels import (
    generate_pseudo_labels, generate_pseudo_labels_augmented,
    confidence_filter, analyze_pseudo_labels
)
from src.train import set_seed, get_device, get_adaptive_batch_size

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')

SEED = 42
DEVICE = get_device('auto')
BATCH_SIZE = get_adaptive_batch_size(DEVICE, default=8)

set_seed(SEED)

## 9.1 Load Best Model and Generate Pseudo-Labels

In [ ]:
# Load the best SSL-pretrained + fine-tuned model
# (Use the best available checkpoint)
model = SegmentationUNet(
    in_channels=1, num_classes=4,
    encoder_channels=[32, 64, 128, 256], dropout=0.1,
)

# Try loading SSL fine-tuned model first, fall back to baseline
ckpt_candidates = [
    os.path.join(CHECKPOINT_DIR, 'ssl_finetune_100pct_best.pth'),
    os.path.join(CHECKPOINT_DIR, 'baseline_100pct_best.pth'),
]

loaded_from = None
for ckpt_path in ckpt_candidates:
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        loaded_from = ckpt_path
        print(f"Loaded model from: {ckpt_path}")
        break

if loaded_from is None:
    raise FileNotFoundError("No trained model checkpoint found!")

model = model.to(DEVICE)
model.eval()

In [ ]:
# Generate pseudo-labels on validation set (for quality analysis)
val_dataset = ACDCSegDataset(
    processed_dir=PROCESSED_DIR,
    split_file=os.path.join(SPLITS_DIR, 'val.json'),
    transform=get_val_transforms(),
)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Generating pseudo-labels on {len(val_dataset)} validation samples...")

# Method 1: Max softmax confidence
pl_results = generate_pseudo_labels(model, val_loader, DEVICE)
print(f"Pseudo-labels shape: {pl_results['pseudo_labels'].shape}")
print(f"Mean confidence: {pl_results['confidence'].mean():.4f}")
print(f"Median confidence: {np.median(pl_results['confidence']):.4f}")

In [ ]:
# Method 2: Augmentation agreement
print("\nGenerating augmented pseudo-labels (5 augmentations)...")
pl_aug_results = generate_pseudo_labels_augmented(
    model, val_loader, DEVICE, n_augmentations=5
)
print(f"Aug. mean confidence: {pl_aug_results['confidence'].mean():.4f}")

## 9.2 Analyze Pseudo-Label Quality

In [ ]:
# Collect ground truth from validation set
all_gt = []
for i in range(len(val_dataset)):
    sample = val_dataset[i]
    all_gt.append(sample['mask'].numpy())
all_gt = np.stack(all_gt)

# Analyze quality
thresholds = [0.0, 0.5, 0.7, 0.8, 0.85, 0.9, 0.95]

analysis = analyze_pseudo_labels(
    pl_results['pseudo_labels'], all_gt,
    pl_results['confidence'], thresholds
)

print("Pseudo-Label Quality Analysis (Max Softmax):")
print(f"{'Threshold':<12} {'Accept %':<12} {'Accuracy':<12}")
print("-" * 36)
for tau in thresholds:
    t_data = analysis['thresholds'][tau]
    print(f"{tau:<12.2f} {t_data['acceptance_rate']*100:<12.1f} {t_data['accuracy']*100:<12.1f}")

print(f"\nOverall accuracy (no threshold): {analysis['overall_accuracy']*100:.1f}%")

In [ ]:
# Plot threshold sweep
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

accept_rates = [analysis['thresholds'][t]['acceptance_rate']*100 for t in thresholds]
accuracies = [analysis['thresholds'][t]['accuracy']*100 for t in thresholds]

axes[0].plot(thresholds, accept_rates, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Confidence Threshold')
axes[0].set_ylabel('Acceptance Rate (%)')
axes[0].set_title('Pseudo-Label Acceptance vs Threshold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, accuracies, 's-', color='coral', linewidth=2)
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Pseudo-Label Accuracy vs Threshold')
axes[1].grid(True, alpha=0.3)

# Confidence histogram
axes[2].hist(pl_results['confidence'].flatten(), bins=50, color='mediumpurple', edgecolor='black')
axes[2].axvline(x=0.9, color='red', linestyle='--', label='τ=0.9')
axes[2].set_xlabel('Confidence')
axes[2].set_ylabel('Pixel Count')
axes[2].set_title('Confidence Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pseudo_label_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10.1 Visualize Before/After Confidence Filtering

In [ ]:
# Visualize pseudo-labels before and after filtering
threshold = 0.9
filtered = confidence_filter(pl_results['pseudo_labels'], pl_results['confidence'], threshold)

print(f"Confidence threshold: {threshold}")
print(f"Acceptance rate: {filtered['acceptance_rate']*100:.1f}%")

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for i in range(4):
    idx = i * (len(val_dataset) // 4)
    sample = val_dataset[idx]
    img = sample['image'].squeeze().numpy()
    gt = sample['mask'].numpy()
    pl = pl_results['pseudo_labels'][idx]
    conf = pl_results['confidence'][idx]
    pl_filt = filtered['filtered_labels'][idx]
    
    axes[i, 0].imshow(img, cmap='gray')
    axes[i, 0].set_title('Image' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(gt, cmap='nipy_spectral', vmin=0, vmax=3)
    axes[i, 1].set_title('Ground Truth' if i == 0 else '')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pl, cmap='nipy_spectral', vmin=0, vmax=3)
    axes[i, 2].set_title('Pseudo-Label (raw)' if i == 0 else '')
    axes[i, 2].axis('off')
    
    # Show filtered (mask out ignored regions)
    pl_vis = np.where(pl_filt >= 0, pl_filt, -1)
    axes[i, 3].imshow(img, cmap='gray', alpha=0.3)
    mask_vis = np.ma.masked_where(pl_vis < 0, pl_vis)
    axes[i, 3].imshow(mask_vis, cmap='nipy_spectral', vmin=0, vmax=3, alpha=0.7)
    axes[i, 3].set_title(f'Filtered (τ={threshold})' if i == 0 else '')
    axes[i, 3].axis('off')

fig.suptitle('Pseudo-Labels: Before and After Confidence Filtering', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pseudo_labels_confidence.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save Table D: Pseudo-Label Analysis
table_d_rows = []
for tau in thresholds:
    t_data = analysis['thresholds'][tau]
    
    # Compute 'Final Dice' placeholder — actual Dice would come from
    # training with these pseudo-labels (done in notebook 07)
    table_d_rows.append({
        'Threshold': tau,
        'Acceptance_pct': f"{t_data['acceptance_rate']*100:.1f}",
        'PL_Accuracy': f"{t_data['accuracy']*100:.1f}",
    })

table_d = pd.DataFrame(table_d_rows)
print("\nTable D: Pseudo-Label Analysis")
print(table_d.to_string(index=False))
table_d.to_csv(os.path.join(TABLES_DIR, 'pseudo_label_analysis.csv'), index=False)

# Save analysis
with open(os.path.join(RESULTS_DIR, 'pseudo_label_analysis.json'), 'w') as f:
    # Convert numpy types for JSON
    def convert(obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, dict):
            return {str(k): convert(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [convert(v) for v in obj]
        return obj
    json.dump(convert(analysis), f, indent=2)

print("\n=== Steps 9-10: Pseudo-Labels + Confidence Filtering COMPLETE ===")